In [10]:
import sys
sys.path.append(r'C:\Users\Stefa\Documents\studio-paper')

from datetime import date
import pandas as pd
from pyspark.sql import functions as F

from parfitt_trb import TRBConfig, run_trb
from parfitt_trb.aggregation import SparkAggregator
from parfitt_trb.local_spark import build_local_spark
from tests.helpers import row, make_sdf

spark = build_local_spark("trb-study")
print(spark.version)


4.1.2


In [11]:
from examples.synth import simulate_panel
import pandas as pd

# Parametri "veri" del dataset sintetico — il modello dovrà recuperarli
K_TRUE    = 0.30   # penetrazione ultima (30% dei category buyer proverà il brand)
A_TRUE    = 0.20   # velocità di diffusione
RBR_TRUE  = 0.25   # repeat buying rate a regime

pdf = simulate_panel(
    n_households=4000, weeks=36,
    K=K_TRUE, a=A_TRUE,
    rbr_start=0.45, rbr_stable=RBR_TRUE,
    cat_interval=2, seed=42,
)
sdf = spark.createDataFrame(pdf)
print(f"Righe totali: {sdf.count()}")
sdf.limit(5).show()


Righe totali: 72000
+----------+----------+--------------+-----------+------+
|shopper_id|  txn_date|is_new_product|is_category|volume|
+----------+----------+--------------+-----------+------+
|        h0|2024-01-03|         false|       true|   1.0|
|        h0|2024-01-17|         false|       true|   1.0|
|        h0|2024-01-31|         false|       true|   1.0|
|        h0|2024-02-14|         false|       true|   1.0|
|        h0|2024-02-28|         false|       true|   1.0|
+----------+----------+--------------+-----------+------+



In [12]:
sdf.groupBy("is_category").agg(F.min("txn_date"), F.max("txn_date")).show()
sdf.groupBy("is_new_product").agg(F.min("txn_date"), F.max("txn_date")).show()


+-----------+-------------+-------------+
|is_category|min(txn_date)|max(txn_date)|
+-----------+-------------+-------------+
|       true|   2024-01-03|   2024-08-28|
+-----------+-------------+-------------+

+--------------+-------------+-------------+
|is_new_product|min(txn_date)|max(txn_date)|
+--------------+-------------+-------------+
|         false|   2024-01-03|   2024-08-28|
|          true|   2024-01-03|   2024-08-28|
+--------------+-------------+-------------+



In [13]:
cfg = TRBConfig(
    launch_date="2024-01-01",
    analysis_date="2024-08-30",
    period_length_days=14,       # intervalli RBR di 2 settimane
    buying_index_base="triers",
    buying_index_window_days=7,
)

res = run_trb(sdf, cfg)

print("=== TRBResult ===")
print(f"trial_index  : {res.trial_index:.4f}   (atteso ~{K_TRUE:.2f})")
print(f"buying_index : {res.buying_index:.4f}   (atteso ~1.0 su dataset sintetico)")
print(f"K stimato    : {res.penetration.ultimate_penetration:.4f}   (atteso ~{K_TRUE:.2f})")
print(f"a stimato    : {res.penetration.growth_rate:.4f}   (atteso ~{A_TRUE:.2f})")
ur = res.ultimate_rbr()
print(f"RBR ultimo   : {ur[1]:.4f} all'intervallo {ur[0]}   (atteso ~{RBR_TRUE:.2f})")
print(f"Share segm.  : {res.segmented_share():.4f}")


=== TRBResult ===
trial_index  : 0.2998   (atteso ~0.30)
buying_index : 1.0000   (atteso ~1.0 su dataset sintetico)
K stimato    : 0.2999   (atteso ~0.30)
a stimato    : 0.2039   (atteso ~0.20)
RBR ultimo   : 0.2706 all'intervallo 17   (atteso ~0.25)
Share segm.  : 0.0795


In [30]:
from parfitt_trb.core import buying_index_from_scopes, buying_index_series

agg = SparkAggregator(sdf, cfg)
scopes = agg.buying_scopes()
bi = buying_index_from_scopes(scopes, "__triers__")
bidx = buying_index_series(agg.buying_series())

print(scopes)
print("=== Buying Index ===")
print(bi)
print("=== Buying Index Series ===")
print(bidx)

           scope  sum_cat  n_buyers
0        __all__   4000.0      4000
1     __triers__   1199.0      1199
2  __repeaters__   1169.0      1169
3           1-6w    759.0       759
4          7-12w    308.0       308
5         13-24w    121.0       121
6           25+w     11.0        11
=== Buying Index ===
1.0
=== Buying Index Series ===
[(1, 1.0), (3, 1.0), (5, 1.0), (7, 1.0), (9, 1.0), (11, 1.0), (13, 1.0), (15, 1.0), (17, 1.0), (19, 1.0), (21, 1.0), (23, 1.0), (25, 1.0), (27, 1.0), (29, 1.0), (31, 1.0), (33, 1.0), (35, 1.0)]


In [29]:
# volume buyer categoria
(
    sdf
    .where(F.col('txn_date') > '2024-08-23')
    .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    .show()
)
# volume categoria buyer prod
carte = (
    sdf
    .where(F.col('txn_date') > '2024-08-23')
    .where(F.col('is_new_product') == True)
    .select('shopper_id').distinct()
    # .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    # .show()
)

(
    sdf
    .where(F.col('txn_date') > '2024-08-23')
    .join(carte, on='shopper_id', how='inner')
    .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    .show()
)

print('all time')
(
    sdf
    .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    .show()
)

(
    sdf
    .join(carte, on='shopper_id', how='inner')
    .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    .show()
)

+-----------+--------------------------+
|sum(volume)|count(DISTINCT shopper_id)|
+-----------+--------------------------+
|     4000.0|                      4000|
+-----------+--------------------------+

+-----------+--------------------------+
|sum(volume)|count(DISTINCT shopper_id)|
+-----------+--------------------------+
|      312.0|                       312|
+-----------+--------------------------+

all time
+-----------+--------------------------+
|sum(volume)|count(DISTINCT shopper_id)|
+-----------+--------------------------+
|    72000.0|                      4000|
+-----------+--------------------------+

+-----------+--------------------------+
|sum(volume)|count(DISTINCT shopper_id)|
+-----------+--------------------------+
|     5616.0|                       312|
+-----------+--------------------------+



In [33]:
# coorte 1-6w

carte = (
    sdf
    .where(F.col('txn_date') <= '2024-02-11')
    .where(F.col('is_new_product') == True)
    .select('shopper_id').distinct()
    # .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    # .show()
)

(
    sdf
    .where(F.col('txn_date') > '2024-08-23')
    .join(carte, on='shopper_id', how='inner')
    .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    .show()
)

+-----------+--------------------------+
|sum(volume)|count(DISTINCT shopper_id)|
+-----------+--------------------------+
|      759.0|                       759|
+-----------+--------------------------+



In [31]:
display(scopes)

print(bi)
print(res.buying_index)

,scope,sum_cat,n_buyers
0,__all__,4000.0,4000
1,__triers__,1199.0,1199
2,__repeaters__,1169.0,1169
3,1-6w,759.0,759
4,7-12w,308.0,308
5,13-24w,121.0,121
6,25+w,11.0,11


1.0
1.0


In [ ]:
# check buying series

display(agg.buying_series())


carte = (
    sdf
    .where(F.col('txn_date') <= '2024-01-28')
    .where(F.col('is_new_product') == True)
    .select('shopper_id').distinct()
    # .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    # .show()
)

(
    sdf
    .where(F.col('txn_date') > '2024-01-14')
    .where(F.col('txn_date') <= '2024-01-28')
    # .where(F.col('is_new_product') == True)
    .join(carte, on='shopper_id', how='inner')
    .agg(F.sum('volume'), F.countDistinct('shopper_id'))
    .show()
)

,period,sel_sum,sel_n,all_sum,all_n
9,1,218.0,218,4000.0,4000
14,3,541.0,541,4000.0,4000
5,5,759.0,759,4000.0,4000
17,7,904.0,904,4000.0,4000
8,9,1002.0,1002,4000.0,4000
10,11,1067.0,1067,4000.0,4000
15,13,1111.0,1111,4000.0,4000
4,15,1140.0,1140,4000.0,4000
13,17,1160.0,1160,4000.0,4000
0,19,1173.0,1173,4000.0,4000


+-----------+--------------------------+
|sum(volume)|count(DISTINCT shopper_id)|
+-----------+--------------------------+
|      411.0|                       411|
+-----------+--------------------------+



```
sdf (Spark)
  └─► SparkAggregator   ← tutto il lavoro pesante gira qui in Spark
          │
          ├── entrants()      → pandas (piccola)
          ├── rbr_pooled()    → pandas (piccola)
          ├── buying_scopes() → pandas (piccola)
          └── share_long()    → pandas (piccola)
                │
                └─► core.py  ← matematica pura, niente Spark
                        └─► TRBResult
```

In [4]:
# Istanziamo l'aggregator direttamente (run_trb lo fa internamente e poi lo chiude)
agg = SparkAggregator(sdf, cfg)

# _prepare() è già stato eseguito in __init__; il risultato è in agg._p
# Lo rieseguiamo sul sdf originale per vederlo
p = agg._prepare(sdf)
p.limit(10).show()

print("\nSchema:")
p.printSchema()

print(f"\nRighe originali : {sdf.count()}")
print(f"Righe dopo _prepare: {p.count()}")

# simulate_panel genera già un pannello pulito senza duplicati — _prepare in questo 
# caso non fa niente di visibile, ma in produzione (dove lo stesso shopper può avere 
# due righe per lo stesso prodotto nella stessa data) è lì che somma le quantità invece 
# di eliminarle.


+----+----------+--------+------+---+
|card|        ts|is_brand|is_cat|qty|
+----+----------+--------+------+---+
|  h0|2024-01-17|   false|  true|1.0|
|  h0|2024-03-27|   false|  true|1.0|
|  h1|2024-02-28|   false|  true|1.0|
|  h1|2024-05-22|   false|  true|1.0|
|  h1|2024-07-03|   false|  true|1.0|
|  h2|2024-01-03|   false|  true|1.0|
|  h2|2024-01-17|   false|  true|1.0|
|  h2|2024-02-14|   false|  true|1.0|
|  h2|2024-04-10|   false|  true|1.0|
|  h2|2024-08-28|   false|  true|1.0|
+----+----------+--------+------+---+


Schema:
root
 |-- card: string (nullable = true)
 |-- ts: date (nullable = true)
 |-- is_brand: boolean (nullable = true)
 |-- is_cat: boolean (nullable = true)
 |-- qty: double (nullable = true)


Righe originali : 72000
Righe dopo _prepare: 72000


In [5]:
# _trials è già in cache dentro agg; lo raccogliamo in pandas per leggerlo
trials = agg._trials.toPandas()
print(f"Trier totali: {len(trials)}")
print(trials.head(10).to_string())
print("\nColonne:", trials.columns.tolist())
print("\nDistribuzione cohort:")
print(trials["cohort"].value_counts().sort_index())


Trier totali: 1199
   card    trial_ts  entry_week  entry_period  cohort  max_interval
0   h28  2024-01-03           1             1    1-6w            19
1   h35  2024-02-14           7             7   7-12w            16
2   h60  2024-03-13          11            11   7-12w            14
3   h75  2024-01-17           3             3    1-6w            18
4   h90  2024-01-17           3             3    1-6w            18
5  h153  2024-04-10          15            15  13-24w            12
6  h158  2024-03-27          13            13  13-24w            13
7  h183  2024-01-03           1             1    1-6w            19
8  h213  2024-03-27          13            13  13-24w            13
9  h216  2024-04-10          15            15  13-24w            12

Colonne: ['card', 'trial_ts', 'entry_week', 'entry_period', 'cohort', 'max_interval']

Distribuzione cohort:
cohort
1-6w      759
13-24w    121
25+w       11
7-12w     308
Name: count, dtype: int64


In [6]:
ent = agg.entrants()
print(ent.to_string())
print(f"\nTotale brand trier  : {ent['n_brand_new'].sum()}")
print(f"Totale cat trier    : {ent['n_cat_new'].sum()}")
print(f"Trial index (ratio) : {ent['n_brand_new'].sum() / ent['n_cat_new'].sum():.4f}")


    period  n_brand_new  n_cat_new
0        1          218       4000
1        3          323          0
2        5          218          0
3        7          145          0
4        9           98          0
5       11           65          0
6       13           44          0
7       15           29          0
8       17           20          0
9       19           13          0
10      21            9          0
11      23            6          0
12      25            4          0
13      27            3          0
14      29            1          0
15      31            2          0
16      35            1          0

Totale brand trier  : 1199
Totale cat trier    : 4000
Trial index (ratio) : 0.2998


In [7]:
from parfitt_trb.core import build_penetration, fit_penetration

pen_raw = build_penetration(
    entrants=ent,
    origin=agg.origin,
    denominator="dynamic",
)

print("Primeri 8 periodi della serie P(t):")
for t, p in pen_raw.series[:8]:
    print(f"  periodo {t:2d}:  P = {p:.4f}  ({p*100:.1f}%  di {ent['n_cat_new'].sum()} cat buyer)")

print(f"\nSnapshot finale: {pen_raw.snapshot:.4f}  (= trial_index)")


Primeri 8 periodi della serie P(t):
  periodo  1:  P = 0.0545  (5.5%  di 4000 cat buyer)
  periodo  2:  P = 0.0545  (5.5%  di 4000 cat buyer)
  periodo  3:  P = 0.1353  (13.5%  di 4000 cat buyer)
  periodo  4:  P = 0.1353  (13.5%  di 4000 cat buyer)
  periodo  5:  P = 0.1898  (19.0%  di 4000 cat buyer)
  periodo  6:  P = 0.1898  (19.0%  di 4000 cat buyer)
  periodo  7:  P = 0.2260  (22.6%  di 4000 cat buyer)
  periodo  8:  P = 0.2260  (22.6%  di 4000 cat buyer)

Snapshot finale: 0.2998  (= trial_index)


In [8]:
pen_fitted = fit_penetration(pen_raw, method="discounted", discount_weight=0.6)

print(f"K stimato : {pen_fitted.ultimate_penetration:.4f}  (vero: {K_TRUE})")
print(f"a stimato : {pen_fitted.growth_rate:.4f}  (vero: {A_TRUE})")
print(f"Nota      : '{pen_fitted.note}'")

# Vediamo cosa fa il fit: costruisce la differenza centrata ΔP(t)
import numpy as np
ps = np.array([p for _, p in pen_raw.series])
dP = (ps[2:] - ps[:-2]) / 2.0   # differenza centrata
x  = ps[1:-1]                    # P(t) come regressore

print("\nPrimi 6 punti della regressione:")
print("   P(t)       ΔP(t)")
for xi, dpi in zip(x[:6], dP[:6]):
    print(f"   {xi:.4f}     {dpi:.4f}")


K stimato : 0.2999  (vero: 0.3)
a stimato : 0.2039  (vero: 0.2)
Nota      : ''

Primi 6 punti della regressione:
   P(t)       ΔP(t)
   0.0545     0.0404
   0.1353     0.0404
   0.1353     0.0272
   0.1898     0.0272
   0.1898     0.0181
   0.2260     0.0181


In [9]:
from parfitt_trb.core import build_penetration, fit_penetration

print(f"{'Settimane':>10}  {'K stimato':>10}  {'a stimato':>10}  {'P osservata':>12}  {'Nota'}")
print("-" * 70)
for cutoff in [8, 12, 16, 20, 24, 28, 32, 36]:
    sub = ent[ent["period"] <= cutoff]
    if sub.empty:
        continue
    p = build_penetration(sub, agg.origin, "dynamic")
    fit_penetration(p, method="discounted", discount_weight=0.6)
    K = f"{p.ultimate_penetration:.4f}" if p.ultimate_penetration else "N/A"
    a = f"{p.growth_rate:.4f}" if p.growth_rate else "N/A"
    obs = f"{p.snapshot:.4f}"
    print(f"{cutoff:>10}  {K:>10}  {a:>10}  {obs:>12}  {p.note[:40]}")


 Settimane   K stimato   a stimato   P osservata  Nota
----------------------------------------------------------------------
         8      0.3281      0.1583        0.2260  
        12      0.3035      0.1846        0.2667  
        16      0.3006      0.1911        0.2850  
        20      0.3002      0.1928        0.2933  
        24      0.3001      0.1931        0.2970  
        28      0.3002      0.1912        0.2988  
        32      0.3000      0.1961        0.2995  
        36      0.2999      0.2039        0.2998  


In [10]:
from parfitt_trb.core import build_penetration, fit_penetration, pwsd

print(f"{'Sett':>5}  {'K':>7}  {'a':>7}  {'P_obs':>7}  {'P_fit':>7}  {'pwsd':>7}")
print("-" * 55)
for cutoff in [8, 12, 16, 20, 24, 28, 32, 36]:
    sub = ent[ent["period"] <= cutoff]
    if sub.empty:
        continue
    p = build_penetration(sub, agg.origin, "dynamic")
    fit_penetration(p, method="discounted", discount_weight=0.6)
    if p.ultimate_penetration is None:
        print(f"{cutoff:>5}  {'N/A':>7}  {'N/A':>7}  {p.snapshot:>7.4f}  {'N/A':>7}  {'N/A':>7}")
        continue
    # curva teorica sui periodi osservati
    fitted_vals = [p.fitted(t) for t, _ in p.series if p.fitted(t) is not None]
    obs_vals    = [v for _, v in p.series[:len(fitted_vals)]]
    sd = pwsd(obs_vals, fitted_vals) if len(obs_vals) >= 2 else float("nan")
    p_fit_last = p.fitted(p.series[-1][0])
    print(f"{cutoff:>5}  {p.ultimate_penetration:>7.4f}  {p.growth_rate:>7.4f}"
          f"  {p.snapshot:>7.4f}  {p_fit_last:>7.4f}  {sd:>7.4f}")


 Sett        K        a    P_obs    P_fit     pwsd
-------------------------------------------------------
    8   0.3281   0.1583   0.2260   0.2198   0.1298
   12   0.3035   0.1846   0.2667   0.2637   0.0546
   16   0.3006   0.1911   0.2850   0.2835   0.0217
   20   0.3002   0.1928   0.2933   0.2925   0.0085
   24   0.3001   0.1931   0.2970   0.2966   0.0034
   28   0.3002   0.1912   0.2988   0.2985   0.0015
   32   0.3000   0.1961   0.2995   0.2993   0.0007
   36   0.2999   0.2039   0.2998   0.2997   0.0004
